#### **Problem Statement**

- Retrieval Augmented Generation is a prompting technique for optimizing the output of a large language model, by - referencing a knowledge source in addition to the training data sources before a response is generated.

- Why is Retrieval Augmented Generation important? : Improves **factual consistency** in response and making the response more trustworthy, addressing the problem of hallucination in generative AI.

- RAG Assessment Tool (RAGAS) is a framework for testing and evaluating QA Pipeline.

- RAG assessment consists of metrics for generation and retrieval.

- Generation - category is divided into faithfulness (is the response grounded in the retrieved context, measures hallucination / factual accuracy) and answer relevance (does the response address the question)

- Retrieval - is divided into Context Precision (signal to noise ratio of retrieved context) and Context Recall (can it retrieve all the relevant information required to provide a response to the question)

- The harmonic mean of these four aspects is the RAGAS score

#### **References :**

[1] https://www.promptingguide.ai/techniques/rag

[2] Lewis, Patrick, et al. "Retrieval-augmented generation for knowledge-intensive nlp tasks." Advances in Neural Information Processing Systems 33 (2020): 9459-9474.

[3] https://aws.amazon.com/what-is/retrieval-augmented-generation/

[4] https://www.kaggle.com/datasets/radek1/sci-or-not-sci-hypthesis-testing-pack

[5] https://arxiv.org/pdf/2309.15217.pdf

[6] https://arxiv.org/pdf/2401.12379.pdf

[7] https://docs.ragas.io/en/latest/howtos/integrations/langchain.html


#### **Installing Libraries**

In [ ]:
# Installing dependencies : Langchain framework, Faiss vector database, tiktoken, datasets, pypdf, ragas, sentence transformers
# Pinned versions (tested Oct 2026) so this notebook keeps working on Google Colab
!pip install -q langchain==0.1.20 langchain-community==0.0.38 langchain-core==0.1.52 langchain-text-splitters==0.0.2 langchain-openai==0.1.7 openai==1.58.1 tiktoken==0.14.0 ragas==0.0.22 datasets==5.0.1 langsmith==0.1.147 numpy==1.26.4 unstructured==0.27.10 pypdf==6.19.0 faiss-cpu==1.15.1 chromadb==0.5.23 sentence-transformers
!sed -i "s/gpt-3.5-turbo-16k/gpt-4o-mini/g" $(python -c "import importlib.util,os;print(os.path.dirname(importlib.util.find_spec('ragas').origin))")/llms/*.py  # ragas 0.0.22 defaults to a retired OpenAI model; point it to gpt-4o-mini
# !pip install langchain -q   # (replaced by the pinned install above)
# !pip install langchain-openai -q   # (replaced by the pinned install above)
# !pip install tiktoken -q   # (replaced by the pinned install above)
# !pip install langchain_community -q   # (replaced by the pinned install above)
# !pip install unstructured -q   # (replaced by the pinned install above)
# !pip install pypdf -q   # (replaced by the pinned install above)
# !pip install faiss-gpu -q   # (replaced by the pinned install above)
# !pip install sentence-transformers -q   # (replaced by the pinned install above)
# !pip install datasets -q   # (replaced by the pinned install above)
# !pip install ragas==0.0.22 -q   # (replaced by the pinned install above)
# !pip install transformers -q   # (replaced by the pinned install above)
# !pip install torch -q   # (replaced by the pinned install above)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.9/815.9 kB 14.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 26.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.2/241.2 kB 13.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.7/226.7 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.9/75.9 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This be

#### **Import required libraries**

In [ ]:
from langchain_community.document_loaders import PyPDFLoader  # Importing PyPDF document loader to load PDF document
from langchain_community.embeddings import HuggingFaceEmbeddings # Hugging face embeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter # Text Splitter
from langchain_community.vectorstores import FAISS # FAISS Vector Database
from langchain_openai import ChatOpenAI #OpenAI interface from Langchain
from langchain_core.runnables import RunnablePassthrough #Allows to pass an input unchanged
from langchain_core.output_parsers.string import StrOutputParser # parses llm output to an appropriate format
from langchain_core.prompts import ChatPromptTemplate # langchain prompt template for chat models

#### **API_KEY**

In [ ]:
# Extracting OPENAI_API_KEY using userdata.get()

from google.colab import userdata
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

In [ ]:
# Set the OpenAI_API_KEY Environment - variable
import os
os.environ['OPENAI_API_KEY'] = "sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

In [ ]:
# Login to Google Account
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#### **Document Loader in Langchain**

- Document loaders : Load the data from a specific source in the form of a document.
(a document consists of text with associated metadata)
- Example: TextLoader, ArxivLoader, PyPDFLoader and more.


- https://python.langchain.com/docs/modules/data_connection/document_loaders/

In [ ]:
# Loading a document using PyPDFLoader()
loader = PyPDFLoader('/content/drive/MyDrive/ref_.pdf')
data = loader.load()

#### **Text Splitter**

- Recursive character text split : splits text by recursively looking at characters.

- https://api.python.langchain.com/en/latest/text_splitter/langchain.text_splitter.RecursiveCharacterTextSplitter.html

In [ ]:
# Recursive Character Text Splitter with chunk size : 500 and chunk overlap 20
# creating meaningful chunks of text with some overlap for ensuring context between the chunks.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 20,
    length_function = len,
    is_separator_regex = False
)

text_split = text_splitter.split_documents(data)

#### **Embeddings**

- The data is broken into meaningful chunks. These need to be converted into a vector representation, which involves transforming the text data into embeddings which are numeric representations that capture the semantic meaning behind text.

- Examples : OpenAI Embeddings, HuggingFaceEmbeddings with specific models such as BERT.

- https://www.datacamp.com/blog/what-is-retrieval-augmented-generation-rag

- https://python.langchain.com/docs/integrations/text_embedding/openai

- https://huggingface.co/sentence-transformers/all-mpnet-base-v2

In [ ]:
# Initializing HuggingFaceEmbeddings() with sentence-transformers text embeddings model
# HuggingFace API_Key from within google colab
embeddings_model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

#### **Vector Store**

- FAISS is a library for efficient similarity search and clustering of dense vectors

- https://python.langchain.com/docs/integrations/vectorstores/faiss


In [ ]:
# FAISS.from_documents() method takes in text chunks with the appropriate embeddings model
db = FAISS.from_documents(text_split, embeddings_model)

#### **Retriever Interface**

- VectorStoreRetriever, backed by a vectorstore.
- When a query enters a system, it must also be converted into embeddings.
- The same model must be used for document and query embedding to ensure uniformity between the two.
- Two search types to retrieve text : Retrieve text based on similarity : uses similarity search in the retriever object, where it selects the text chunk vectors that are most similar (using cosine similarity or Euclidean distance) to the question vector.
- "MMR", maximum marginal relevance search, optimizing for similarity to query AND diversity among the selected documents.

- https://towardsdatascience.com/4-ways-of-question-answering-in-langchain-188c6707cc5


In [ ]:
# Vector Store Retriever
retriever = db.as_retriever(search_type = "similarity", search_kwargs= {"k": 5})

#### **Prompt for Retrieval Augmented Generation**

In [ ]:
# Providing a prompt to the large language model to guide the model response with
# context

prompt_template = """ Human : Answer honestly based on the context provided.
              Otherwise, say you don't know. Please do not make up an answer.

              ### CONTEXT
               {context}

              ###  QUESTION
               {question}


               \n


               AI :

               """

In [ ]:
# ChatOpenAI() method with the specified model : gpt-4o-mini model
# ChatPromptTemplate.from_template() method with the prompt template provided to guide the model response

model = ChatOpenAI(openai_api_key = OPENAI_API_KEY, model_name = "gpt-4o-mini")
prompt = ChatPromptTemplate.from_template(prompt_template)

In [ ]:
# retrieval_chain for QA with the retreiver interface provided as context and the specified large language
# model
# The question is passed unchanged using RunnablePassthrough() method when the retrieval chain is invoked.


retrieval_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()}
    | prompt
    | model
    |StrOutputParser()
)

response = retrieval_chain.invoke("What is LoRa? How is the effectiveness of LoRA Training")

In [ ]:
# response from the model

response

'LoRA stands for low-rank adapters, and it is a fine-tuning method that modifies only a fraction of the total model weights to reduce memory footprint during training. The effectiveness of LoRA training is nearly as effective as full model fine-tuning, as stated by Dettmers et al. 2023.'

#### **Evaluation of the response using RAGAS**

- **Answer Relevancy** : How relevant is the generated answer to the question.
- **Faithfulness** : Factual accuracy of the generated answer.
- **Context Recall** : Quantifies if all of the relevant information required to answer the question was retrieved.

In [ ]:
# importing Dataset from datasets library
from datasets import Dataset

In [ ]:
# Question and ground truth pairs for RAG Evaluation

questions = ["What is a notable obeservation regarding SQL queries incorrectly flagged as erroneous?",
             "What does the enhanced spider dataset for fine-tuning include?",
             "A portion of incorrectly generated SQL queries fall into what category?"
             ]

ground_truths = [["A notable observation is that many SQL queries, incorrectly flagged as erroneous, actually produce result tables identical to those generated by the ground truth SQL"],
                 ["The enhanced spider dataset for fine-tuning includes the database schema for each entry, along with the skeleton format in the responses"],
                 ["A significant portion of the SQL queries incorrectly generated by the LLMs fall into a category characterized by their unnecessary complexity and deviation from the simplicity of the ground truth solutions"]
]

In [ ]:
# Inference : Preparation of the remaining information (extracting answers from the model
# and the context
answers = []
contexts = []
for query in questions:
  answers.append(retrieval_chain.invoke(query))
  contexts.append([docs.page_content for docs in retriever.get_relevant_documents(query)])

In [ ]:
# Creating a dictionary consisting of questions, extracted answers, context and the ground truth
data = {
    "question":questions,
    "answer":answers,
    "contexts":contexts,
    "ground_truths":ground_truths
}

In [ ]:
# Answers extracted using the retrieval_chain.invoke() method
answers

['A notable observation is that many SQL queries, incorrectly flagged as erroneous, actually produce result tables identical to those generated by the ground truth SQL.',
 'The enhanced spider dataset for fine-tuning includes the database schema for each entry, along with the skeleton format in the responses.',
 'A significant portion of incorrectly generated SQL queries fall into a category characterized by unnecessary complexity and deviation from the simplicity of the ground truth solutions.']

In [ ]:
# Creating a dataset for evaluation using the RAGAS Framework
dataset = Dataset.from_dict(data)

In [ ]:
# Importing dependencies for evaluation using the RAGAS Framework
from ragas import evaluate
from ragas.metrics import(
    faithfulness,
    answer_relevancy,
    context_recall,
)

In [ ]:
# Metrics to evaluate the response generated using retrieval_chain.invoke() method
result = evaluate(
    dataset=dataset,
    metrics=[
        context_recall,
        faithfulness,
        answer_relevancy
    ]
)

evaluating with [context_recall]


100%|██████████| 1/1 [00:01<00:00,  1.80s/it]


evaluating with [faithfulness]


100%|██████████| 1/1 [00:05<00:00,  5.35s/it]


evaluating with [answer_relevancy]


100%|██████████| 1/1 [00:02<00:00,  2.68s/it]


In [ ]:
# Displaying the RAGAS Metrics for each response in the form of a Pandas DataFrame
df = result.to_pandas()
df

,question,answer,contexts,ground_truths,context_recall,faithfulness,answer_relevancy
0,What is a notable obeservation regarding SQL q...,A notable observation is that many SQL queries...,"[flawed query structure, any post-processing s...",[A notable observation is that many SQL querie...,1.0,1.0,0.919997
1,What does the enhanced spider dataset for fine...,The enhanced spider dataset for fine-tuning in...,[Figure 1: Shows the open-source fine-tuning d...,[The enhanced spider dataset for fine-tuning i...,1.0,1.0,1.000000
2,A portion of incorrectly generated SQL queries...,A significant portion of incorrectly generated...,"[flawed query structure, any post-processing s...",[A significant portion of the SQL queries inco...,1.0,1.0,0.975738
